In [14]:
!pip install qiskit qiskit-aer qiskit-nature qiskit-algorithms pyscf matplotlib pandas --quiet

In [15]:
import numpy as np
import pandas as pd
from pyscf import gto, scf, mcscf, cc

HARTREE_TO_EV = 27.211386245988
EXPERIMENTAL_PI_PISTAR_EV = 7.6
np.random.seed(1)

In [16]:
def rotate_z(xyz, angle_deg):
    a = np.radians(angle_deg)
    R = np.array([[np.cos(a), -np.sin(a), 0],
                  [np.sin(a),  np.cos(a), 0],
                  [0, 0, 1]])
    return R @ np.array(xyz)

def build_geometry(twist_deg: float = 0.0) -> str:
    atoms = {k: v.copy() for k, v in equilibrium_atoms.items()}
    if twist_deg != 0.0:
        atoms["H3"] = rotate_z(atoms["H3"], twist_deg)
        atoms["H4"] = rotate_z(atoms["H4"], twist_deg)
    order = ["C1", "C2", "H1", "H2", "H3", "H4"]
    return "; ".join(f"{name[0]} {x:.6f} {y:.6f} {z:.6f}" for name in order
                      for x, y, z in [atoms[name]])

def print_geometry(label, geometry):
    print(f"{label}:")
    atoms = geometry.split("; ")
    for i, atom in enumerate(atoms):
        suffix = ";" if i < len(atoms) - 1 else ""
        print(f"  {atom}{suffix}")

# Standard planar equilibrium ethylene geometry (Angstrom)
equilibrium_atoms = {
    "C1": np.array([0.0000,  0.0000,  0.6695]),
    "C2": np.array([0.0000,  0.0000, -0.6695]),
    "H1": np.array([0.0000,  0.9289,  1.2321]),
    "H2": np.array([0.0000, -0.9289,  1.2321]),
    "H3": np.array([0.0000,  0.9289, -1.2321]),
    "H4": np.array([0.0000, -0.9289, -1.2321]),
}


geometry_equilibrium = build_geometry(0.0)
geometry_twisted = build_geometry(90.0)

# print(geometry_equilibrium)
print_geometry("Equilibrium", geometry_equilibrium)
print_geometry("\nTwisted 90", geometry_twisted)

Equilibrium:
  C 0.000000 0.000000 0.669500;
  C 0.000000 0.000000 -0.669500;
  H 0.000000 0.928900 1.232100;
  H 0.000000 -0.928900 1.232100;
  H 0.000000 0.928900 -1.232100;
  H 0.000000 -0.928900 -1.232100

Twisted 90:
  C 0.000000 0.000000 0.669500;
  C 0.000000 0.000000 -0.669500;
  H 0.000000 0.928900 1.232100;
  H 0.000000 -0.928900 1.232100;
  H -0.928900 0.000000 -1.232100;
  H 0.928900 -0.000000 -1.232100


In [21]:
mol = gto.M(atom=geometry_equilibrium.replace("; ", "\n"), basis="sto-3g",
            charge=0, spin=0, unit="Angstrom")
mf = scf.RHF(mol).run(verbose=0)

mc_casci = mcscf.CASCI(mf, 2, 2)
mc_casci.verbose = 0
e_casci = mc_casci.kernel()[0]

mc_casscf = mcscf.CASSCF(mf, 2, 2)
mc_casscf.verbose = 0
e_casscf = mc_casscf.kernel()[0]

mc_sa = mcscf.CASCI(mf, 2, 2)
mc_sa.verbose = 0
mc_sa.fcisolver.nstates = 2
e_states_casci = mc_sa.kernel()[0]
casci_excitation_eV = (e_states_casci[1] - e_states_casci[0]) * HARTREE_TO_EV

mycc = cc.CCSD(mf).run(verbose=0)
eom_excitation_eV = np.array(mycc.eomee_ccsd_singlet(nroots=2)[0]) * HARTREE_TO_EV

print(f"RHF energy : {mf.e_tot:.6f} Ha")
print(f"CASCI(2,2) ground energy : {e_casci:.6f} Ha")
print(f"CASSCF(2,2) ground energy : {e_casscf:.6f} Ha  (orbital-optimized)")
print(f"CASCI(2,2) S0->S1 excitation : {casci_excitation_eV:.3f} eV")
print(f"EOM-CCSD (full space) excitations : {np.round(eom_excitation_eV, 3)} eV")

RHF energy : -77.072088 Ha
CASCI(2,2) ground energy : -77.116593 Ha
CASSCF(2,2) ground energy : -77.116599 Ha  (orbital-optimized)
CASCI(2,2) S0->S1 excitation : 4.792 eV
EOM-CCSD (full space) excitations : [11.609 12.921] eV
